# Tier 4: Coupled Mamba against the BioLangFusion heads

**The question.** All three BioLangFusion heads are positionwise in one way or another. Concatenation mixes the three modalities at each codon independently. The gated head picks one weighting for the whole sequence. Cross-attention mixes positions by content but carries no state from one codon to the next. None of them can let one modality change how a *later* codon of another modality is read.

Coupled Mamba can (Li et al., NeurIPS 2024). Each modality runs its own selective state-space block, and at every codon each modality's transition is applied to the pooled state of all three. Information accumulates along the codon axis while the modalities stay coupled. The implementation and the project's four deviations from the paper are documented in `shared_code/fusion/coupled_mamba.py`.

**What this notebook tests.** If fusion gains come from long-range context, where one modality changes how another should be read further along the sequence, Coupled Mamba should beat the positionwise heads. If the useful signal is local, the modalities agreeing or disagreeing codon by codon, it should land on concatenation and add nothing.

**Three designs, so a result can be attributed:**

| Design | What it is | Isolates |
| --- | --- | --- |
| `coupled_mamba` | Projections to width 128, two coupled Mamba layers, fusion to 600 | |
| `uncoupled_mamba` | The same layers and parameter count, each modality's state on its own chain | the coupling, against `coupled_mamba` |
| `mamba_bottleneck` | The projections to width 128 and the final fusion, no Mamba layers | the recurrence, against `uncoupled_mamba` |

**Matched to the references.** Every fit uses the Phase A recipe exactly: BioLangFusion's optimizer and stopping settings, 100 TextCNN channels, the Equation 1 grid, seed 0, the published splits and BioLangFusion's own encoders (NT v2 100M, RNA-FM, ESM-2 8M). Only the fusion head differs. The references are the saved seed-0 fits in `results/phaseA/` (and Stage 3's for fungal), which carry per-row test predictions, so every comparison is a paired bootstrap on the same rows.

**Default Run All reads saved results only** and reports what is missing. Training starts only when `READ_SAVED_RESULTS = False`.

In [ ]:
# Environment, Python module paths, elapsed time and filesystem paths.
import os
import sys
import time
from pathlib import Path

# Set before importing torch: unsupported Apple/MPS operations may fall back to CPU.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import numpy as np
import pandas as pd
import torch
from IPython.display import display

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

In [ ]:
# The experiment's device preference, as in Phase A: Apple/MPS, NVIDIA/CUDA, then CPU.
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print("PyTorch:", torch.__version__, "| device:", DEVICE)

In [ ]:
# Find Code/shared_code from the repository root or from inside Code.
working_dir = Path.cwd().resolve()
for parent in (working_dir, *working_dir.parents):
    if (parent / "shared_code").is_dir():
        CODE_DIR = parent
        break
    if (parent / "Code" / "shared_code").is_dir():
        CODE_DIR = parent / "Code"
        break
else:
    raise FileNotFoundError("Cannot locate Code/shared_code. Start this notebook from within the repository.")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from shared_code.fusion import blf, coupled_mamba, heads, train

print("Registered fusion heads:", sorted(heads.FUSIONS))

In [ ]:
# Default Run All reads stored results only; it never trains.
READ_SAVED_RESULTS = True

# All four benchmarks with saved references. To start with one, shorten this tuple.
BENCHMARKS = ("ecoli_proteins", "cov_vac_degradation", "fungal_expression", "mrna_stability")

# Cheapest first: the bottleneck has no Mamba layers; the uncoupled design runs three scans.
DESIGNS = ("mamba_bottleneck", "coupled_mamba", "uncoupled_mamba")

# Inputs and references are read from Stages 3 and 4; this experiment writes only here.
CACHE_ROOT = CODE_DIR / "stage3" / "embeddings"
RESULTS_ROOT = CODE_DIR / "stage4" / "results" / "tier4_designs"
PHASE_A = CODE_DIR / "stage4" / "results" / "phaseA"
# Searched in order. Fungal's Phase A references come from Stage 3, except the grid-RNA one.
REFERENCE_DIRS = {
    "cov_vac_degradation": (PHASE_A / "cov_vac_degradation",),
    "ecoli_proteins": (PHASE_A / "ecoli_proteins",),
    "mrna_stability": (PHASE_A / "mrna_stability",),
    "fungal_expression": (CODE_DIR / "stage3" / "results" / "fungal_expression",
                          PHASE_A / "fungal_expression"),
}

# Phase A's four eligible single-encoder baselines; the best is chosen on validation.
SINGLE_VARIANTS = ("unimodal:dna", "unimodal:rna", "unimodal:prot", "unimodal-grid:rna")

# Phase A's recipe exactly: the trainer's BioLangFusion defaults, 100 channels, Equation 1, seed 0.
CONFIG = train.TrainConfig(channels=100, tconv="equation", seed=0)
print(CONFIG)

## Predictions, written before any fit

Write down what each design should do on each benchmark, with Chase, **before** the first fit, and do not edit this cell afterward. A prediction written after the result is not a test of the interpretation.

The proposal below follows the project's reading of each benchmark. Replace it with your own if you disagree.

| Benchmark | What drives the target, by our reading | Prediction |
| --- | --- | --- |
| E. coli expression | Codon usage, close to a bag of codons | Local. `coupled_mamba` ties `concat`; coupling adds nothing over `uncoupled_mamba` |
| Fungal expression | Composition (protein and codon), per the swap results | Local. Same as E. coli |
| mRNA stability | Codon optimality, also largely compositional | Local. Same as E. coli |
| CoV-Vac degradation | Chemical degradation of short constructs, set by which bases are paired, a long-range property | Long-range. `coupled_mamba` above `concat` and above `uncoupled_mamba` |

If Coupled Mamba wins only where predicted, the interpretation holds. If it wins everywhere, the positionwise heads were missing something more general than structure. If it wins nowhere, the useful signal on these benchmarks is local, which is consistent with the Phase B finding that the encoder matters more than the head.

## Train the designs

Skipped unless `READ_SAVED_RESULTS = False`. A completed fit is found on disk and reused, never refitted. Each fit should take roughly as long as a cross-attention fit in Phase A.

In [ ]:
if READ_SAVED_RESULTS:
    print("READ_SAVED_RESULTS is True: no training. Set it to False to fill missing fits.")
else:
    for benchmark in BENCHMARKS:
        started = time.time()
        streams = blf.BenchmarkStreams.open(benchmark, CACHE_ROOT)
        try:
            for design in DESIGNS:
                train.run(streams, design, CONFIG, DEVICE, RESULTS_ROOT / benchmark)
        finally:
            streams.close()
        print(f"{benchmark}: {(time.time() - started) / 60:.1f} min")

## Read the designs and the references

Scores are Spearman correlation for the regression benchmarks and accuracy (percent) for E. coli. The best single encoder is chosen on validation among Phase A's four candidates: the three encoders at their native resolution and the RNA stream read after the codon grid. Using the same set keeps "beats the best single encoder" meaning what it means in the deck.

In [ ]:
def tag(benchmark, variant, tconv=None):
    return (f"{benchmark}__{variant}__lambda0.0__ch{CONFIG.channels}__{tconv or CONFIG.tconv}"
            f"__seed{CONFIG.seed}.json")


def load(path):
    return train.Result.load(path) if path.exists() else None


def load_reference(benchmark, variant):
    # A native single-encoder model never touches the codon grid, so its grid label is
    # immaterial: prefer the matching label, else the historical appendix one (fungal's
    # RNA and protein references exist only under that label), as Chase's swap reader does.
    # Everything that uses the grid must match the Equation 1 label exactly.
    name = variant.replace(":", "-")
    labels = (None, "appendix") if variant.startswith("unimodal:") else (None,)
    for label in labels:
        for folder in REFERENCE_DIRS[benchmark]:
            found = load(folder / tag(benchmark, name, tconv=label))
            if found is not None:
                return found
    return None


runs, missing = {}, []
for benchmark in BENCHMARKS:
    found = {}
    for variant in ("concat", "cross", *SINGLE_VARIANTS):
        found[variant] = load_reference(benchmark, variant)
    for design in DESIGNS:
        found[design] = load(RESULTS_ROOT / benchmark / tag(benchmark, design))
    singles = {v: found[v] for v in SINGLE_VARIANTS if found[v] is not None}
    if singles:
        best = max(singles, key=lambda v: singles[v].val_score)
        found["best single"] = singles[best]
        found["best single name"] = best
    missing += [f"{benchmark} / {v}" for v, r in found.items() if r is None]
    runs[benchmark] = found

scale = {b: 100.0 if b == "ecoli_proteins" else 1.0 for b in BENCHMARKS}
table = pd.DataFrame({
    b: {v: (r.test_score * scale[b] if r is not None else np.nan)
        for v, r in runs[b].items() if v not in ("best single name",)}
    for b in BENCHMARKS}).round(3)
display(table)
print("Best single encoder on validation:",
      {b: runs[b].get("best single name", "none") for b in BENCHMARKS})
print("Missing:", missing if missing else "none")

## The contrasts that answer the question

Each row is a paired bootstrap on the same test rows: the first run minus the second, with a 95% interval. **Resolved** means the interval excludes zero. Read them in this order:

1. **Bottleneck minus concat.** Does squeezing each modality to width 128 cost anything? If so, every Mamba design starts behind and the later contrasts should be read with that in mind.
2. **Uncoupled minus bottleneck.** Does carrying state along the codons help, before any coupling?
3. **Coupled minus uncoupled.** Does coupling the modalities' states help, beyond the recurrence? This is the contrast the design exists for.
4. **Coupled minus concat** and **coupled minus cross.** Where does Coupled Mamba stand against the heads we already have?
5. **Coupled minus best single.** The project's standing question: does fusion beat the best single encoder at all?

In [ ]:
CONTRASTS = (
    ("mamba_bottleneck", "concat"),
    ("uncoupled_mamba", "mamba_bottleneck"),
    ("coupled_mamba", "uncoupled_mamba"),
    ("coupled_mamba", "concat"),
    ("coupled_mamba", "cross"),
    ("coupled_mamba", "best single"),
)

rows = []
for benchmark in BENCHMARKS:
    found = runs[benchmark]
    task = "classification" if benchmark == "ecoli_proteins" else "regression"
    for first, second in CONTRASTS:
        a, b = found.get(first), found.get(second)
        if a is None or b is None:
            rows.append({"benchmark": benchmark, "contrast": f"{first} - {second}", "note": "missing"})
            continue
        try:
            c = train.paired_bootstrap(a, b, task, draws=2000, seed=0)
        except ValueError as error:
            rows.append({"benchmark": benchmark, "contrast": f"{first} - {second}", "note": str(error)})
            continue
        s = scale[benchmark]
        rows.append({
            "benchmark": benchmark,
            "contrast": f"{first} - {second}",
            "difference": round(c["difference"] * s, 3),
            "ci_low": round(c["ci_low"] * s, 3),
            "ci_high": round(c["ci_high"] * s, 3),
            "resolved": bool(c["ci_low"] > 0 or c["ci_high"] < 0),
            "note": "",
        })
display(pd.DataFrame(rows))